# RFP RAG 연동 실험

Vector DB 기반 RAG 검색 결과를 Generation 프롬프트에 연결하고, Golden QA 기준으로 소형 3종 모델의 응답 품질을 비교합니다.

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path

# 공유받은 Chroma DB가 들어 있어야 하는 경로
chroma_dir = Path("./data/chroma_db")

print("현재 작업 경로:", Path.cwd())
print("Chroma DB 경로 존재 여부:", chroma_dir.exists())

if chroma_dir.exists():
    print("\nChroma DB 내부 파일 목록:")
    for p in chroma_dir.rglob("*"):
        print(p)
else:
    print("\n./data/chroma_db 경로가 없습니다.")
    print("공유받은 chroma_db 폴더를 ./data/chroma_db 위치에 넣어야 합니다.")

In [ ]:
# 2단계: Vector DB 및 Retriever 로드

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = "./data/chroma_db"
COLLECTION_NAME = "rfp_docs"

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Vector DB 로드 완료")
print("Retriever 생성 완료")

In [ ]:
# 3단계: Retriever 검색 결과 확인

query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)

print(f"검색 결과 수: {len(docs)}")

for i, doc in enumerate(docs, start=1):
    print("=" * 80)
    print(f"[검색 결과 {i}]")
    print("\nmetadata:")
    print(doc.metadata)
    print("\npage_content 앞부분:")
    print(doc.page_content[:1000])

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
# 4단계: RAG context builder 작성

def build_rag_context(docs, max_content_chars=1200):
    """
    retriever가 반환한 Document 리스트를
    LLM 프롬프트에 넣기 좋은 context 문자열로 변환합니다.
    
    page_content만 사용하지 않고,
    사업명, 발주기관, 사업금액, 사업요약, 문서 위치 정보를 함께 포함합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액: {meta.get("사업금액", "")}
공개일자: {meta.get("공개일자", "")}
입찰참여시작일: {meta.get("입찰참여시작일", "")}
입찰참여마감일: {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

본문:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
context = build_rag_context(docs)

print(context[:4000])

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.

    핵심 원칙:
    1. page_content만 사용하지 않고 metadata를 함께 포함합니다.
    2. 사업명, 발주기관, 사업금액, 사업요약을 상단에 배치합니다.
    3. 검색된 본문 chunk가 질문과 직접 관련 없을 수 있으므로 문서 위치를 함께 제공합니다.
    4. context 길이 증가를 막기 위해 본문은 일부만 사용합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액: {meta.get("사업금액", "")}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
context = build_rag_context(docs)
print(context[:4000])

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
import requests
import time

def ask_ollama(prompt, model, temperature=0.0, timeout=180):
    url = "http://localhost:11434/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    
    start_time = time.time()
    
    try:
        response = requests.post(url, json=payload, timeout=timeout)
        elapsed_time = time.time() - start_time
        
        if response.status_code == 200:
            result = response.json()
            return {
                "success": True,
                "model": model,
                "answer": result.get("response", "").strip(),
                "elapsed_time": elapsed_time,
                "error": None
            }
        else:
            return {
                "success": False,
                "model": model,
                "answer": "",
                "elapsed_time": elapsed_time,
                "error": f"HTTP {response.status_code}: {response.text}"
            }
    except Exception as e:
        elapsed_time = time.time() - start_time
        return {
            "success": False,
            "model": model,
            "answer": "",
            "elapsed_time": elapsed_time,
            "error": str(e)
        }

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 원문 숫자를 임의로 억/만원 단위로 바꾸지 마세요.
4. 사업금액 metadata와 사업요약이 있으면 함께 참고하세요.
5. 검색된 본문이 질문과 직접 관련 없더라도, metadata에 명시된 정보는 사용할 수 있습니다.
6. 한국어 존댓말로 간결하게 답변하세요.
7. 답변 마지막에 사용한 근거의 문서명 또는 사업명을 간단히 적어주세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변]
""".strip()

In [ ]:
query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)
context = build_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

1. 환경 설정 및 Vector DB 로드

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 반드시 context에 나온 원문 숫자 표기 그대로 답변하세요.
4. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
5. 사업금액 metadata와 사업요약이 있으면 함께 참고하세요.
6. 검색된 본문이 질문과 직접 관련 없더라도, metadata에 명시된 정보는 사용할 수 있습니다.
7. 한국어 존댓말로 간결하게 답변하세요.
8. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 근거를 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [ ]:
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def format_amount(value):
    """
    metadata의 사업금액이 140000000.0처럼 float로 들어오는 경우
    LLM이 읽기 좋은 원문형 숫자 표기로 변환합니다.
    """
    if value is None or value == "":
        return ""

    try:
        return f"{int(float(value)):,}원"
    except:
        return str(value)

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.
    금액은 LLM이 환산하지 않도록 140,000,000원 형태로 정리합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액_원문표기: {formatted_amount}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 반드시 `사업금액_원문표기`에 있는 값을 그대로 복사해서 답변하세요.
4. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
5. `1억`, `천만`, `만원`, `억 원` 같은 환산 표현은 절대 사용하지 마세요.
6. 한국어 존댓말로 간결하게 답변하세요.
7. 답변은 반드시 아래 형식을 지키세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: 이 사업의 예산은 {{사업금액_원문표기}}입니다.
근거: {{사업명}} / {{문서명}}

[답변]
""".strip()

In [ ]:
query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)
context = build_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

In [ ]:
# 특정 문서 doc_id로 검색 범위를 고정한 retriever
target_doc_id = "D016"

filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"doc_id": target_doc_id}
    }
)

print("Filtered retriever 생성 완료")

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 사용자 질문에 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 사용자의 질문에 직접 답하세요.
4. 질문이 예산/금액에 관한 경우, 금액은 반드시 context의 원문 숫자 표기를 그대로 사용하세요.
5. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
6. 질문이 기간에 관한 경우, 사업요약이나 문서에 있는 기간 표현을 그대로 사용하세요.
7. 질문이 계약 방식에 관한 경우, 계약 방식만 간결하게 답하세요.
8. 질문이 사업 범위에 관한 경우, 사업 범위에 해당하는 항목만 정리하세요.
9. 한국어 존댓말로 간결하게 답변하세요.
10. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [ ]:
rag_test_cases = [
    {
        "case_id": "RAG-01",
        "question": "통합정보시스템 고도화 용역의 예산은 얼마인가요?",
        "expected": "140,000,000원",
        "check_point": "사업금액 원문 표기 유지"
    },
    {
        "case_id": "RAG-02",
        "question": "통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?",
        "expected": "5개월 이내",
        "check_point": "사업기간 추출"
    },
    {
        "case_id": "RAG-03",
        "question": "통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?",
        "expected": "협상에 의한 계약",
        "check_point": "계약 방식 추출"
    },
    {
        "case_id": "RAG-04",
        "question": "통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?",
        "expected": "기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",
        "check_point": "사업 범위 요약"
    },
    {
        "case_id": "RAG-05",
        "question": "통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?",
        "expected": "총 24개",
        "check_point": "요구사항 수 추출"
    },
]

In [ ]:
rag_results = []

model = "exaone3.5:2.4b"

for case in rag_test_cases:
    query = case["question"]
    
    docs = filtered_retriever.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)
    
    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180
    )
    
    rag_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 사용자 질문에 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 사용자의 질문에 직접 답하세요.
4. 질문이 예산/금액에 관한 경우, 금액은 반드시 context의 원문 숫자 표기를 그대로 사용하세요.
5. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
6. 질문이 기간에 관한 경우, `5개월 이내`, `계약일로부터 24개월 이내`처럼 context에 나온 기간 표현을 그대로 사용하세요. `이내`, `이상`, `미만`, `계약일로부터` 같은 표현을 생략하지 마세요.
7. 질문이 계약 방식에 관한 경우, 계약 방식만 간결하게 답하세요.
8. 질문이 사업 범위에 관한 경우, 사업 범위에 해당하는 항목만 정리하세요.
9. 한국어 존댓말로 간결하게 답변하세요.
10. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [ ]:
rag_candidate_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

In [ ]:
rag_compare_results = []

for model in rag_candidate_models:
    print(f"테스트 중: {model}")

    for case in rag_test_cases:
        query = case["question"]

        docs = filtered_retriever.invoke(query)
        context = build_rag_context(docs)
        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180
        )

        rag_compare_results.append({
            "case_id": case["case_id"],
            "model": model,
            "question": query,
            "expected": case["expected"],
            "check_point": case["check_point"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "error": result["error"]
        })

rag_compare_df = pd.DataFrame(rag_compare_results)
rag_compare_df

In [ ]:
rag_compare_df[
    [
        "case_id",
        "model",
        "question",
        "expected",
        "check_point",
        "success",
        "elapsed_time",
        "answer"
    ]
]

In [ ]:
rag_speed_summary = (
    rag_compare_df
    .groupby("model")
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min")
    )
    .reset_index()
)

rag_speed_summary["avg_elapsed_time"] = rag_speed_summary["avg_elapsed_time"].round(2)
rag_speed_summary

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 검색 결과만 근거로 사용자 질문에 답변하세요.

규칙:
- 문서에 있는 정보만 사용하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 사용자의 질문에 직접 답하세요.
- 금액은 원문 숫자 표기 그대로 사용하세요.
- 금액을 억 원, 만 원, 1억 4천만 원처럼 환산하지 마세요.
- 기간은 context에 나온 표현을 그대로 사용하세요. 예: 5개월 이내
- 한국어 존댓말로 간결하게 답변하세요.
- 마지막 줄에는 근거를 `근거: 사업명 / 문서명` 형태로 작성하세요.

검색 결과:
{context}

사용자 질문:
{question}

위 질문에 대한 답변을 작성하세요.
""".strip()

In [ ]:
gemma_retry_results = []

model = "gemma3:4b"

for case in rag_test_cases:
    query = case["question"]

    docs = filtered_retriever.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180
    )

    gemma_retry_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

gemma_retry_df = pd.DataFrame(gemma_retry_results)
gemma_retry_df

In [ ]:
def build_short_rag_context(docs, max_content_chars=300):
    """
    gemma3:4b처럼 긴 context에서 출력이 불안정한 모델을 위한
    짧은 RAG context builder입니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업금액: {formatted_amount}
사업요약:
{meta.get("사업요약", "")}

문서 위치: {meta.get("header_path", "")}

본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
gemma_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {"doc_id": "D016"}
    }
)

print("gemma용 retriever 생성 완료")

In [ ]:
query = "통합정보시스템 고도화 용역의 예산은 얼마인가요?"

docs = gemma_retriever.invoke(query)
context = build_short_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

gemma_rag_test = ask_ollama(
    prompt=prompt,
    model="gemma3:4b",
    temperature=0.0,
    timeout=180,
    num_predict=512
)

print("success:", gemma_rag_test["success"])
print("elapsed_time:", round(gemma_rag_test["elapsed_time"], 2))
print("error:", gemma_rag_test["error"])
print("answer repr:", repr(gemma_rag_test["answer"]))
print("\nanswer:")
print(gemma_rag_test["answer"])

In [ ]:
gemma_short_results = []

model = "gemma3:4b"

for case in rag_test_cases:
    query = case["question"]

    docs = gemma_retriever.invoke(query)
    context = build_short_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    gemma_short_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

gemma_short_df = pd.DataFrame(gemma_short_results)
gemma_short_df

In [ ]:
# 기존 3개 모델 비교 결과에서 gemma3:4b의 비정상 결과 제거
rag_compare_fixed_df = rag_compare_df[rag_compare_df["model"] != "gemma3:4b"].copy()

# 짧은 context로 재실험한 gemma3:4b 결과 추가
rag_compare_fixed_df = pd.concat(
    [rag_compare_fixed_df, gemma_short_df],
    ignore_index=True
)

# 보기 좋게 정렬
rag_compare_fixed_df = rag_compare_fixed_df.sort_values(
    by=["model", "case_id"]
).reset_index(drop=True)

rag_compare_fixed_df[
    [
        "case_id",
        "model",
        "question",
        "expected",
        "check_point",
        "success",
        "elapsed_time",
        "answer"
    ]
]

In [ ]:
rag_speed_summary_fixed = (
    rag_compare_fixed_df
    .groupby("model")
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min")
    )
    .reset_index()
)

rag_speed_summary_fixed["avg_elapsed_time"] = rag_speed_summary_fixed["avg_elapsed_time"].round(2)
rag_speed_summary_fixed

In [ ]:
# 기존 조건: k=5
gemma_default_retriever_k5 = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"doc_id": "D016"}
    }
)

# 중간 조건: k=2
gemma_default_retriever_k2 = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {"doc_id": "D016"}
    }
)

print("gemma 재테스트용 retriever 준비 완료")

In [ ]:
def run_gemma_rag_retest(case, retriever_obj, context_builder, context_setting):
    query = case["question"]

    docs = retriever_obj.invoke(query)
    context = context_builder(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model="gemma3:4b",
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    return {
        "case_id": case["case_id"],
        "model": "gemma3:4b",
        "context_setting": context_setting,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "answer_len": len(result["answer"]),
        "error": result["error"]
    }

In [ ]:
gemma_retest_results = []

for case in rag_test_cases:
    # A. 기존 context + k=5
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k5,
            context_builder=build_rag_context,
            context_setting="default_context_k5"
        )
    )

    # B. 기존 context + k=2
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k2,
            context_builder=build_rag_context,
            context_setting="default_context_k2"
        )
    )

    # C. 짧은 context + k=2
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k2,
            context_builder=build_short_rag_context,
            context_setting="short_context_k2"
        )
    )

gemma_retest_df = pd.DataFrame(gemma_retest_results)

gemma_retest_df[
    [
        "case_id",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

In [ ]:
gemma_retest_summary = (
    gemma_retest_df
    .groupby("context_setting")
    .agg(
        test_count=("case_id", "count"),
        avg_elapsed_time=("elapsed_time", "mean"),
        avg_answer_len=("answer_len", "mean"),
        min_answer_len=("answer_len", "min"),
        max_answer_len=("answer_len", "max")
    )
    .reset_index()
)

gemma_retest_summary[
    ["avg_elapsed_time", "avg_answer_len", "min_answer_len", "max_answer_len"]
] = gemma_retest_summary[
    ["avg_elapsed_time", "avg_answer_len", "min_answer_len", "max_answer_len"]
].round(2)

gemma_retest_summary

2. RAG context 구성 및 초기 테스트

In [ ]:
existing_qa_rag_cases = [
    {
        "case_id": "IQA-RAG-01",
        "original_case_id": "IQA-01",
        "source_type": "existing_qa_adapted",
        "category": "금액 단위 보존",
        "question": "통합정보시스템 고도화 용역의 예산은 얼마인가요?",
        "expected": "140,000,000원",
        "check_points": ["140,000,000원", "억/조/만 원 단위 임의 환산 금지"]
    },
    {
        "case_id": "IQA-RAG-02",
        "original_case_id": "IQA-02",
        "source_type": "existing_qa_adapted",
        "category": "항목 누락 방지",
        "question": "통합정보시스템 고도화 용역의 주요 사업 범위를 모두 알려주세요.",
        "expected": "기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리, 전자결재서식 연동, 연구노트 관리",
        "check_points": ["주요 사업 범위", "항목 누락 최소화", "항목명 왜곡 없음"]
    },
    {
        "case_id": "IQA-RAG-03",
        "original_case_id": "IQA-03",
        "source_type": "existing_qa_adapted",
        "category": "가격점수 가드레일",
        "question": "통합정보시스템 고도화 용역의 가격점수를 계산해주세요.",
        "expected": "계산 불가 또는 정보 부족",
        "check_points": ["계산하지 않음", "입찰가격", "예정가격", "평점산식", "정보 부족"]
    },
    {
        "case_id": "IQA-RAG-04",
        "original_case_id": "IQA-04",
        "source_type": "existing_qa_adapted",
        "category": "가격점수 산식 일부 부족",
        "question": "통합정보시스템 고도화 용역의 입찰가격이 95,000,000원이라고 할 때 가격점수를 계산해주세요.",
        "expected": "계산 불가 또는 정보 부족",
        "check_points": ["계산하지 않음", "예정가격", "평점산식", "정보 부족"]
    },
    {
        "case_id": "IQA-RAG-05",
        "original_case_id": "IQA-05",
        "source_type": "existing_qa_adapted",
        "category": "기간 원문 보존",
        "question": "통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?",
        "expected": "5개월 이내",
        "check_points": ["5개월 이내", "이내 표현 생략 금지"]
    },
    {
        "case_id": "IQA-RAG-06",
        "original_case_id": "IQA-06",
        "source_type": "existing_qa_adapted",
        "category": "문서 근거 부족",
        "question": "통합정보시스템 고도화 용역의 유지보수 인력은 몇 명 투입해야 하나요?",
        "expected": "문서 근거 부족",
        "check_points": ["문서 근거 부족", "추측하지 않음", "인원 임의 생성 금지"]
    },
]

existing_qa_rag_df = pd.DataFrame(existing_qa_rag_cases)
existing_qa_rag_df

In [ ]:
existing_qa_rag_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

existing_qa_rag_results = []

for model in existing_qa_rag_models:
    print(f"기존 QA 기반 RAG 테스트 실행 중: {model}")

    for case in existing_qa_rag_cases:
        query = case["question"]

        if model == "gemma3:4b":
            # gemma3:4b는 k=5에서 출력이 한 글자로 끊기는 현상이 있어 k=2 조건 사용
            docs = gemma_default_retriever_k2.invoke(query)
            context = build_rag_context(docs)
            context_setting = "default_context_k2"
        else:
            docs = filtered_retriever.invoke(query)
            context = build_rag_context(docs)
            context_setting = "default_context_k5"

        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180,
            num_predict=512
        )

        existing_qa_rag_results.append({
            "case_id": case["case_id"],
            "original_case_id": case["original_case_id"],
            "source_type": case["source_type"],
            "category": case["category"],
            "model": model,
            "context_setting": context_setting,
            "question": query,
            "expected": case["expected"],
            "check_points": case["check_points"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "answer_len": len(result["answer"]),
            "error": result["error"]
        })

existing_qa_rag_results_df = pd.DataFrame(existing_qa_rag_results)

existing_qa_rag_results_df[
    [
        "case_id",
        "original_case_id",
        "category",
        "model",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

In [ ]:
existing_qa_rag_speed_summary_df = (
    existing_qa_rag_results_df
    .groupby(["model", "context_setting"])
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min"),
        avg_answer_len=("answer_len", "mean")
    )
    .reset_index()
)

existing_qa_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
] = existing_qa_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
].round(2)

existing_qa_rag_speed_summary_df

In [ ]:
# 세종테크노파크 문서 doc_id 확인
sejong_docs = vectorstore.similarity_search("세종테크노파크 인사정보 전산시스템 구축 용역", k=10)

print("세종테크노파크 검색 결과")
for i, doc in enumerate(sejong_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

In [ ]:
# 경희대학교 문서 doc_id 확인
kyunghee_docs = vectorstore.similarity_search("경희대학교 산학협력단 정보시스템 운영 용역업체 선정", k=10)

print("경희대학교 검색 결과")
for i, doc in enumerate(kyunghee_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

In [ ]:
# 고려대학교 문서 doc_id 확인
korea_docs = vectorstore.similarity_search("고려대학교 차세대 포털 학사 정보시스템 구축사업", k=10)

print("고려대학교 검색 결과")
for i, doc in enumerate(korea_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

In [ ]:
# Vector DB에 들어있는 고유 문서 목록 확인
all_data = vectorstore.get(include=["metadatas"])

unique_docs = {}

for meta in all_data["metadatas"]:
    doc_id = meta.get("doc_id")
    if doc_id not in unique_docs:
        unique_docs[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "사업유형": meta.get("사업유형"),
            "사업금액": meta.get("사업금액"),
        }

unique_docs_df = pd.DataFrame(unique_docs.values()).sort_values("doc_id").reset_index(drop=True)
unique_docs_df

In [ ]:
doc_id_map = {
    "sejong": "D088",
    "kyunghee": "D014",
    "korea": "D008",
}

doc_retrievers = {
    doc_key: vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 5,
            "filter": {"doc_id": doc_id}
        }
    )
    for doc_key, doc_id in doc_id_map.items()
}

# gemma3:4b는 k=5에서 출력 불안정이 확인되어 k=2 retriever를 별도로 생성
doc_retrievers_k2 = {
    doc_key: vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 2,
            "filter": {"doc_id": doc_id}
        }
    )
    for doc_key, doc_id in doc_id_map.items()
}

print("3개 임시 Golden QA 문서 retriever 생성 완료")

In [ ]:
temp_golden_rag_cases = [
    # 세종테크노파크
    {
        "case_id": "TG-RAG-01",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "기본 정보 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 무엇인가요?",
        "expected": "세종테크노파크 인사정보 전산시스템 구축 용역",
        "check_points": ["용역명", "사업명"]
    },
    {
        "case_id": "TG-RAG-02",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "기간/금액 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간과 사업비는 어떻게 되나요?",
        "expected": "사업기간, 사업비",
        "check_points": ["사업기간", "사업비", "금액 원문 표기"]
    },
    {
        "case_id": "TG-RAG-03",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "계약 방식 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 무엇인가요?",
        "expected": "계약방법",
        "check_points": ["계약방법", "협상", "입찰"]
    },

    # 경희대학교
    {
        "case_id": "TG-RAG-04",
        "doc_key": "kyunghee",
        "source_type": "temp_golden_rag",
        "category": "기간/예산 추출",
        "question": "경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?",
        "expected": "사업기간, 예산",
        "check_points": ["사업기간", "예산", "금액 원문 표기"]
    },
    {
        "case_id": "TG-RAG-05",
        "doc_key": "kyunghee",
        "source_type": "temp_golden_rag",
        "category": "운영/유지관리 범위",
        "question": "경희대학교 산학협력단 정보시스템 운영 용역의 주요 운영 및 유지관리 범위는 무엇인가요?",
        "expected": "정보시스템 운영 및 유지관리 범위",
        "check_points": ["운영", "유지관리", "업무 범위"]
    },

    # 고려대학교
    {
        "case_id": "TG-RAG-06",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "기본 정보 추출",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업명은 무엇인가요?",
        "expected": "차세대 포털·학사 정보시스템 구축사업",
        "check_points": ["사업명"]
    },
    {
        "case_id": "TG-RAG-07",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "구축 대상 추출",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 구축 대상 시스템 목록을 알려주세요.",
        "expected": "포털, 학사 정보시스템 등",
        "check_points": ["구축 대상", "시스템 목록", "항목 누락 방지"]
    },
    {
        "case_id": "TG-RAG-08",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "금액 단위 보존",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마인가요?",
        "expected": "금액 원문 표기",
        "check_points": ["금액", "원문 표기", "억/조 단위 임의 환산 금지"]
    },
]

temp_golden_rag_cases_df = pd.DataFrame(temp_golden_rag_cases)
temp_golden_rag_cases_df

In [ ]:
temp_golden_rag_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

temp_golden_rag_results = []

for model in temp_golden_rag_models:
    print(f"임시 Golden QA RAG 테스트 실행 중: {model}")

    for case in temp_golden_rag_cases:
        query = case["question"]
        doc_key = case["doc_key"]

        if model == "gemma3:4b":
            retriever_obj = doc_retrievers_k2[doc_key]
            context_setting = "doc_filtered_k2"
        else:
            retriever_obj = doc_retrievers[doc_key]
            context_setting = "doc_filtered_k5"

        docs = retriever_obj.invoke(query)
        context = build_rag_context(docs)
        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180,
            num_predict=512
        )

        temp_golden_rag_results.append({
            "case_id": case["case_id"],
            "doc_key": doc_key,
            "source_type": case["source_type"],
            "category": case["category"],
            "model": model,
            "context_setting": context_setting,
            "question": query,
            "expected": case["expected"],
            "check_points": case["check_points"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "answer_len": len(result["answer"]),
            "error": result["error"]
        })

temp_golden_rag_results_df = pd.DataFrame(temp_golden_rag_results)

temp_golden_rag_results_df[
    [
        "case_id",
        "doc_key",
        "category",
        "model",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

In [ ]:
for _, row in temp_golden_rag_results_df.iterrows():
    print("=" * 100)
    print("case_id:", row["case_id"])
    print("doc_key:", row["doc_key"])
    print("model:", row["model"])
    print("context_setting:", row["context_setting"])
    print("category:", row["category"])
    print("question:", row["question"])
    print("expected:", row["expected"])
    print("elapsed_time:", row["elapsed_time"])
    print("answer_len:", row["answer_len"])
    print("\nanswer:")
    print(row["answer"])
    print()

In [ ]:
temp_golden_rag_speed_summary_df = (
    temp_golden_rag_results_df
    .groupby(["model", "context_setting"])
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min"),
        avg_answer_len=("answer_len", "mean")
    )
    .reset_index()
)

temp_golden_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
] = temp_golden_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
].round(2)

temp_golden_rag_speed_summary_df

3. metadata 보완

In [ ]:
def format_amount(value):
    """
    metadata의 사업금액을 원문형 숫자 표기로 변환합니다.
    단, 0 또는 결측값은 실제 사업금액이 아니라 추출 실패일 수 있으므로
    금액으로 단정하지 않고 'metadata 미확인'으로 처리합니다.
    """
    if value is None or value == "":
        return "metadata 미확인"

    try:
        amount = int(float(value))
        if amount == 0:
            return "metadata 미확인"
        return f"{amount:,}원"
    except:
        return str(value)

In [ ]:
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.
    metadata의 사업금액이 0 또는 결측값이면 실제 예산으로 단정하지 않습니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액_원문표기: {formatted_amount}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 검색 결과만 근거로 사용자 질문에 답변하세요.

규칙:
- 문서에 있는 정보만 사용하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 사용자의 질문에 직접 답하세요.
- 금액은 원문 숫자 표기 그대로 사용하세요.
- 금액을 억 원, 만 원, 1억 4천만 원처럼 환산하지 마세요.
- metadata의 사업금액이 `metadata 미확인`, `0`, `0원`이면 이를 실제 예산으로 단정하지 마세요.
- metadata 금액이 미확인인 경우, 본문 또는 사업요약에 명시된 `사업예산`, `사업비`, `추정가격` 표현을 우선 확인하세요.
- 기간은 context에 나온 표현을 그대로 사용하세요. 예: 5개월 이내
- 한국어 존댓말로 간결하게 답변하세요.
- 마지막 줄에는 근거를 `근거: 사업명 / 문서명` 형태로 작성하세요.

검색 결과:
{context}

사용자 질문:
{question}

위 질문에 대한 답변을 작성하세요.
""".strip()

In [ ]:
target_case = [case for case in temp_golden_rag_cases if case["case_id"] == "TG-RAG-04"][0]

target_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

metadata_fix_check_results = []

for model in target_models:
    query = target_case["question"]
    doc_key = target_case["doc_key"]

    if model == "gemma3:4b":
        retriever_obj = doc_retrievers_k2[doc_key]
        context_setting = "doc_filtered_k2"
    else:
        retriever_obj = doc_retrievers[doc_key]
        context_setting = "doc_filtered_k5"

    docs = retriever_obj.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    metadata_fix_check_results.append({
        "case_id": target_case["case_id"],
        "model": model,
        "context_setting": context_setting,
        "question": query,
        "expected": target_case["expected"],
        "answer": result["answer"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer_len": len(result["answer"]),
        "error": result["error"]
    })

metadata_fix_check_df = pd.DataFrame(metadata_fix_check_results)
metadata_fix_check_df

In [ ]:
for _, row in metadata_fix_check_df.iterrows():
    print("=" * 100)
    print("model:", row["model"])
    print("context_setting:", row["context_setting"])
    print("question:", row["question"])
    print("expected:", row["expected"])
    print("elapsed_time:", row["elapsed_time"])
    print("\nanswer:")
    print(row["answer"])
    print()

In [ ]:
# TG-RAG-04에서 실제로 검색된 context 확인
target_case = [case for case in temp_golden_rag_cases if case["case_id"] == "TG-RAG-04"][0]

query = target_case["question"]
doc_key = target_case["doc_key"]

docs = doc_retrievers[doc_key].invoke(query)
context = build_rag_context(docs)

print(context[:6000])

In [ ]:
# D014 경희대학교 문서 전체 chunk에서 예산/기간 관련 키워드 검색

kyunghee_data = vectorstore.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

keywords = [
    "400,000,000",
    "200,000,000",
    "2026.04.30",
    "2026. 04. 30",
    "사업예산",
    "예산",
    "사업기간",
    "계약체결일",
    "부가세",
    "부가가치세"
]

matched_chunks = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    text = doc_text or ""
    hit_keywords = [kw for kw in keywords if kw in text]
    
    if hit_keywords:
        matched_chunks.append({
            "hit_keywords": hit_keywords,
            "section_id": meta.get("section_id"),
            "block_id": meta.get("block_id"),
            "header_path": meta.get("header_path"),
            "text": text[:1500]
        })

print("매칭 chunk 수:", len(matched_chunks))

for i, item in enumerate(matched_chunks[:20], start=1):
    print("=" * 100)
    print(f"[매칭 결과 {i}]")
    print("hit_keywords:", item["hit_keywords"])
    print("section_id:", item["section_id"])
    print("block_id:", item["block_id"])
    print("header_path:", item["header_path"])
    print("\ntext:")
    print(item["text"])

In [ ]:
# D014 경희대학교 문서 전체 chunk에서 더 넓은 키워드로 재검색

kyunghee_data = vectorstore.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

keywords = [
    "400",
    "400,000",
    "400000",
    "400,000,000",
    "금400",
    "금 400",
    "200,000",
    "200000",
    "200,000,000",
    "금200",
    "금 200",
    "2026",
    "2026.04",
    "2026. 04",
    "2026.04.30",
    "2026. 04. 30",
    "계약체결",
    "계약 체결",
    "용역기간",
    "사업기간",
    "사업예산",
    "예산액",
    "부가세 포함",
    "부가가치세 포함",
]

matched_chunks_wide = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    text = doc_text or ""
    hit_keywords = [kw for kw in keywords if kw in text]
    
    if hit_keywords:
        matched_chunks_wide.append({
            "hit_keywords": hit_keywords,
            "section_id": meta.get("section_id"),
            "block_id": meta.get("block_id"),
            "header_path": meta.get("header_path"),
            "text": text[:2000]
        })

print("넓은 키워드 매칭 chunk 수:", len(matched_chunks_wide))

for i, item in enumerate(matched_chunks_wide[:30], start=1):
    print("=" * 100)
    print(f"[매칭 결과 {i}]")
    print("hit_keywords:", item["hit_keywords"])
    print("section_id:", item["section_id"])
    print("block_id:", item["block_id"])
    print("header_path:", item["header_path"])
    print("\ntext:")
    print(item["text"])

In [ ]:
# D014 문서의 전체 chunk 위치 확인
kyunghee_chunk_index = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    kyunghee_chunk_index.append({
        "section_id": meta.get("section_id"),
        "block_id": meta.get("block_id"),
        "header_path": meta.get("header_path"),
        "text_preview": (doc_text or "")[:120]
    })

kyunghee_chunk_index_df = pd.DataFrame(kyunghee_chunk_index)

pd.set_option("display.max_colwidth", None)
kyunghee_chunk_index_df[
    ["section_id", "block_id", "header_path", "text_preview"]
].head(80)

In [ ]:
# 전체 문서 기준 metadata 품질 점검

all_data = vectorstore.get(include=["documents", "metadatas"])

metadata_audit = {}

for doc_text, meta in zip(all_data["documents"], all_data["metadatas"]):
    doc_id = meta.get("doc_id")
    
    if doc_id not in metadata_audit:
        raw_amount = meta.get("사업금액", None)
        
        try:
            amount_value = int(float(raw_amount)) if raw_amount not in [None, ""] else None
        except:
            amount_value = None
        
        metadata_audit[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "사업유형": meta.get("사업유형"),
            "사업금액": raw_amount,
            "사업금액_숫자": amount_value,
            "입찰참여시작일": meta.get("입찰참여시작일"),
            "입찰참여마감일": meta.get("입찰참여마감일"),
            "chunk_count": 0,
        }
    
    metadata_audit[doc_id]["chunk_count"] += 1

metadata_audit_df = pd.DataFrame(metadata_audit.values())

metadata_audit_df.head()

In [ ]:
# 사업금액이 0, 결측, 미확인인 문서 확인

amount_problem_df = metadata_audit_df[
    metadata_audit_df["사업금액_숫자"].isna() |
    (metadata_audit_df["사업금액_숫자"] == 0)
].copy()

amount_problem_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "사업금액",
        "사업금액_숫자",
        "chunk_count"
    ]
].sort_values("doc_id")

In [ ]:
# 입찰참여기간이 unknown인 문서 확인

period_problem_df = metadata_audit_df[
    metadata_audit_df["입찰참여시작일"].astype(str).str.contains("unknown|None|nan", case=False, na=True) |
    metadata_audit_df["입찰참여마감일"].astype(str).str.contains("unknown|None|nan", case=False, na=True)
].copy()

period_problem_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "입찰참여시작일",
        "입찰참여마감일",
        "chunk_count"
    ]
].sort_values("doc_id")

In [ ]:
import re

amount_pattern = re.compile(
    r"(\d{1,3}(,\d{3})+|\d+)\s*원|금\s*\d|사업예산|사업비|추정가격|부가세|부가가치세"
)

date_pattern = re.compile(
    r"\d{4}\.\s*\d{1,2}\.\s*\d{1,2}|계약\s*체결|계약체결|사업기간|용역기간|계약기간"
)

doc_text_audit = {}

for doc_text, meta in zip(all_data["documents"], all_data["metadatas"]):
    doc_id = meta.get("doc_id")
    text = doc_text or ""
    
    if doc_id not in doc_text_audit:
        doc_text_audit[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "has_amount_like_text": False,
            "has_date_like_text": False,
            "amount_hit_count": 0,
            "date_hit_count": 0,
        }
    
    if amount_pattern.search(text):
        doc_text_audit[doc_id]["has_amount_like_text"] = True
        doc_text_audit[doc_id]["amount_hit_count"] += 1
    
    if date_pattern.search(text):
        doc_text_audit[doc_id]["has_date_like_text"] = True
        doc_text_audit[doc_id]["date_hit_count"] += 1

doc_text_audit_df = pd.DataFrame(doc_text_audit.values())

doc_quality_audit_df = metadata_audit_df.merge(
    doc_text_audit_df[
        [
            "doc_id",
            "has_amount_like_text",
            "has_date_like_text",
            "amount_hit_count",
            "date_hit_count"
        ]
    ],
    on="doc_id",
    how="left"
)

doc_quality_audit_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "사업금액",
        "사업금액_숫자",
        "has_amount_like_text",
        "amount_hit_count",
        "has_date_like_text",
        "date_hit_count"
    ]
].sort_values("doc_id")

In [ ]:
# metadata는 문제인데 본문에는 금액/기간 표현이 있는 문서
# → metadata 추출 실패 가능성이 큰 문서

suspicious_metadata_df = doc_quality_audit_df[
    (
        doc_quality_audit_df["사업금액_숫자"].isna() |
        (doc_quality_audit_df["사업금액_숫자"] == 0)
    ) &
    (doc_quality_audit_df["has_amount_like_text"] == True)
].copy()

suspicious_metadata_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "사업금액",
        "사업금액_숫자",
        "amount_hit_count",
        "date_hit_count"
    ]
].sort_values(["amount_hit_count", "date_hit_count"], ascending=False)

---

In [ ]:
from pathlib import Path
import tarfile
import time
import chromadb
import os
import sys

PROJECT_DIR = Path.cwd()
ARCHIVE_PATH = PROJECT_DIR / "vector_db.tar.gz"
DATA_DIR = PROJECT_DIR / "data"
EXTRACT_DIR = DATA_DIR / f"linux_fresh_vector_test_{int(time.time())}"

print("현재 작업 디렉토리:", os.getcwd())
print("python:", sys.executable)
print("platform:", sys.platform)
print("chromadb:", chromadb.__version__)

print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("압축파일 존재:", ARCHIVE_PATH.exists())

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"vector_db.tar.gz를 찾지 못했습니다: {ARCHIVE_PATH}")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE_PATH, "r:gz") as tar_ref:
    tar_ref.extractall(EXTRACT_DIR)

print("압축 해제 위치:", EXTRACT_DIR)

sqlite_paths = list(EXTRACT_DIR.rglob("chroma.sqlite3"))
print("chroma.sqlite3 개수:", len(sqlite_paths))

for p in sqlite_paths:
    print("찾은 DB:", p)

if len(sqlite_paths) == 0:
    raise FileNotFoundError("chroma.sqlite3를 찾지 못했습니다.")

CHROMA_DIR = sqlite_paths[0].parent
print("CHROMA_DIR:", CHROMA_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collections = client.list_collections()
print("collections:", collections)

for col in collections:
    print("collection name:", col.name)

collection = client.get_collection("rfp_docs")

try:
    print("count:", collection.count())
    print("count 성공")
except Exception as e:
    print("count 실패")
    print(type(e).__name__)
    print(e)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name="rfp_docs",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

data = vectorstore._collection.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

docs = data["documents"]
metas = data["metadatas"]

print("D014 chunk 수:", len(docs))

keywords = [
    "사업기간",
    "계약기간",
    "계약체결일",
    "2026.04.30",
    "2026",
    "사업예산",
    "예산",
    "400,000,000",
    "400000000",
    "금400",
    "부가가치세",
]

for kw in keywords:
    matched = []

    for i, text in enumerate(docs):
        if kw in text:
            matched.append(i)

    print("=" * 100)
    print("키워드:", kw)
    print("포함 chunk 수:", len(matched))

    for idx in matched[:5]:
        print("-" * 80)
        print("chunk index:", idx)
        print(metas[idx])
        print(docs[idx][:1500])

In [ ]:
from pathlib import Path
import tarfile
import time
import chromadb
import os
import sys

PROJECT_DIR = Path.cwd()
ARCHIVE_PATH = PROJECT_DIR / "vector_db_v2.tar.gz"
DATA_DIR = PROJECT_DIR / "data"
EXTRACT_DIR = DATA_DIR / f"linux_fresh_vector_v2_test_{int(time.time())}"

print("현재 작업 디렉토리:", os.getcwd())
print("python:", sys.executable)
print("platform:", sys.platform)
print("chromadb:", chromadb.__version__)

print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("압축파일 존재:", ARCHIVE_PATH.exists())

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"vector_db_v2.tar.gz를 찾지 못했습니다: {ARCHIVE_PATH}")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE_PATH, "r:gz") as tar_ref:
    tar_ref.extractall(EXTRACT_DIR)

print("압축 해제 위치:", EXTRACT_DIR)

sqlite_paths = list(EXTRACT_DIR.rglob("chroma.sqlite3"))
print("chroma.sqlite3 개수:", len(sqlite_paths))

for p in sqlite_paths:
    print("찾은 DB:", p)

if len(sqlite_paths) == 0:
    raise FileNotFoundError("chroma.sqlite3를 찾지 못했습니다.")

CHROMA_DIR = sqlite_paths[0].parent
print("CHROMA_DIR:", CHROMA_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collections = client.list_collections()
print("collections:", collections)

for col in collections:
    print("collection name:", col.name)

collection = client.get_collection("rfp_docs")

try:
    print("count:", collection.count())
    print("count 성공")
except Exception as e:
    print("count 실패")
    print(type(e).__name__)
    print(e)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name="rfp_docs",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

data = vectorstore._collection.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

docs = data["documents"]
metas = data["metadatas"]

print("D014 chunk 수:", len(docs))

keywords = [
    "사업기간",
    "계약기간",
    "계약체결일",
    "2026.04.30",
    "2026",
    "사업예산",
    "예산",
    "400,000,000",
    "400000000",
    "금400",
    "부가가치세",
]

for kw in keywords:
    matched = []

    for i, text in enumerate(docs):
        if kw in text:
            matched.append(i)

    print("=" * 100)
    print("키워드:", kw)
    print("포함 chunk 수:", len(matched))

    for idx in matched[:5]:
        print("-" * 80)
        print("chunk index:", idx)
        print(metas[idx])
        print(docs[idx][:1500])

In [ ]:
data = vectorstore._collection.get(
    include=["metadatas"]
)

metas = data["metadatas"]

doc_map = {}

for m in metas:
    doc_id = m.get("doc_id")
    file_name = m.get("file_name")
    사업명 = m.get("사업명")
    발주기관 = m.get("발주기관")
    사업금액 = m.get("사업금액")
    시작일 = m.get("입찰참여시작일")
    마감일 = m.get("입찰참여마감일")

    if doc_id not in doc_map:
        doc_map[doc_id] = {
            "file_name": file_name,
            "사업명": 사업명,
            "발주기관": 발주기관,
            "사업금액": 사업금액,
            "입찰참여시작일": 시작일,
            "입찰참여마감일": 마감일,
            "count": 0,
        }

    doc_map[doc_id]["count"] += 1

for doc_id, info in sorted(doc_map.items()):
    print("=" * 100)
    print("doc_id:", doc_id)
    print("chunk 수:", info["count"])
    print("file_name:", info["file_name"])
    print("사업명:", info["사업명"])
    print("발주기관:", info["발주기관"])
    print("사업금액:", info["사업금액"])
    print("입찰참여시작일:", info["입찰참여시작일"])
    print("입찰참여마감일:", info["입찰참여마감일"])

In [ ]:
def check_doc_keywords(doc_id, keywords):
    data = vectorstore._collection.get(
        where={"doc_id": doc_id},
        include=["documents", "metadatas"]
    )

    docs = data["documents"]
    metas = data["metadatas"]

    print("=" * 100)
    print("doc_id:", doc_id)
    print("chunk 수:", len(docs))

    if metas:
        print("file_name:", metas[0].get("file_name"))
        print("사업명:", metas[0].get("사업명"))
        print("발주기관:", metas[0].get("발주기관"))
        print("사업금액:", metas[0].get("사업금액"))
        print("입찰참여시작일:", metas[0].get("입찰참여시작일"))
        print("입찰참여마감일:", metas[0].get("입찰참여마감일"))

    for kw in keywords:
        matched = []

        for i, text in enumerate(docs):
            if kw in text:
                matched.append(i)

        print("-" * 100)
        print("키워드:", kw)
        print("포함 chunk 수:", len(matched))

        for idx in matched[:3]:
            print("chunk index:", idx)
            print("metadata:", metas[idx])
            print(docs[idx][:800])

In [ ]:
check_doc_keywords(
    doc_id="D014",
    keywords=[
        "사업기간",
        "계약기간",
        "사업예산",
        "예산",
        "400000000",
        "400,000,000",
        "2026",
    ]
)

4. 다중문서 검색 전략 개선

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("현재 사용할 CHROMA_DIR:", CHROMA_DIR)

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name="rfp_docs",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print("vectorstore / retriever 재생성 완료")

In [ ]:
data = vectorstore._collection.get(
    where={"doc_id": "D014"},
    include=["metadatas"],
    limit=3
)

for i, meta in enumerate(data["metadatas"], 1):
    print("=" * 80)
    print(i)
    print("file_name:", meta.get("file_name"))
    print("사업금액:", meta.get("사업금액"))
    print("입찰참여시작일:", meta.get("입찰참여시작일"))
    print("입찰참여마감일:", meta.get("입찰참여마감일"))

In [ ]:
def format_money(value):
    try:
        if value in [None, "", "<unknown>"]:
            return "<unknown>"
        return f"{int(float(value)):,}원"
    except:
        return str(value)


def format_rag_context(docs):
    context_blocks = []

    for i, doc in enumerate(docs, 1):
        meta = doc.metadata

        business_amount = format_money(meta.get("사업금액", "<unknown>"))

        block = f"""
[검색결과 {i}]
[문서명] {meta.get("file_name", "<unknown>")}
[사업명] {meta.get("사업명", "<unknown>")}
[발주기관] {meta.get("발주기관", "<unknown>")}
[사업금액] {business_amount}
[입찰참여시작일] {meta.get("입찰참여시작일", "<unknown>")}
[입찰참여마감일] {meta.get("입찰참여마감일", "<unknown>")}
[섹션] {meta.get("header_path", "<unknown>")}
[문서내용]
{doc.page_content}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [ ]:
query = "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업금액은 얼마인가요?"

retrieved_docs = retriever.invoke(query)
context = format_rag_context(retrieved_docs)

print(context[:3000])

In [ ]:
def build_rag_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색된 문서 근거]만 사용하여 질문에 답변하세요.

[검색된 문서 근거]
{context}

[질문]
{question}

[답변 작성 규칙]
- 반드시 한국어 존댓말로 답변하세요.
- 문서 근거에 있는 내용만 사용하세요.
- 질문에서 묻지 않은 사업금액, 일정, 발주기관 정보는 답변에 포함하지 마세요.
- [사업금액] 항목이 있고 질문이 금액을 묻는 경우에만 그 값을 그대로 복사해서 답변하세요.
- 금액을 억 원, 만 원 단위로 환산하지 마세요.
- 예: [사업금액] 400,000,000원 이면 답변에도 반드시 "400,000,000원"이라고 쓰세요.
- <unknown>으로 표시된 항목은 알 수 없는 정보로 처리하세요.
- 근거가 부족한 경우 추정하지 말고, 확인 가능한 근거가 부족하다고 답변하세요.
- "가능성이 높습니다", "추정됩니다", "일 것으로 보입니다"처럼 근거 없는 추정 표현을 사용하지 마세요.
"""

In [ ]:
question = "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업금액은 얼마인가요?"

retrieved_docs = retriever.invoke(question)
context = format_rag_context(retrieved_docs)
prompt = build_rag_prompt(question, context)

print(prompt[:4000])

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
import requests
import time

def ask_ollama(model, prompt, temperature=0.1, num_predict=1024):
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
        }
    }

    start_time = time.time()

    try:
        response = requests.post(url, json=payload, timeout=300)

        if response.status_code != 200:
            return {
                "model": model,
                "answer": f"HTTP {response.status_code} 오류: {response.text}",
                "elapsed_sec": None
            }

        result = response.json()
        elapsed = time.time() - start_time

        return {
            "model": model,
            "answer": result.get("response", "").strip(),
            "elapsed_sec": round(elapsed, 2)
        }

    except requests.exceptions.ConnectionError:
        return {
            "model": model,
            "answer": "Ollama 서버에 연결하지 못했습니다. `ollama serve` 실행 여부를 확인하세요.",
            "elapsed_sec": None
        }

    except Exception as e:
        return {
            "model": model,
            "answer": f"오류 발생: {type(e).__name__} - {e}",
            "elapsed_sec": None
        }

In [ ]:
model = "qwen2.5:3b"

answer = ask_ollama(model, prompt)

print(answer)

In [ ]:
question = "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업금액은 얼마인가요?"

retrieved_docs = retriever.invoke(question)
context = format_rag_context(retrieved_docs)
prompt = build_rag_prompt(question, context)

result = ask_ollama("qwen2.5:3b", prompt)

print(result)

In [ ]:
question = "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업기간은 언제까지인가요?"

retrieved_docs = retriever.invoke(question)
context = format_rag_context(retrieved_docs)
prompt = build_rag_prompt(question, context)

result = ask_ollama("qwen2.5:3b", prompt)

print(result)

In [ ]:
test_models = [
    "qwen2.5:3b",
    "gemma3:4b",
    "exaone3.5:2.4b",
]

test_questions = [
    "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업금액은 얼마인가요?",
    "경희대학교 산학협력단 정보시스템 운영 용역업체 선정의 사업기간은 언제까지인가요?",
]

for model in test_models:
    print("=" * 100)
    print("MODEL:", model)

    for question in test_questions:
        retrieved_docs = retriever.invoke(question)
        context = format_rag_context(retrieved_docs)
        prompt = build_rag_prompt(question, context)

        result = ask_ollama(model, prompt)

        print("-" * 100)
        print("질문:", question)
        print("응답시간:", result["elapsed_sec"])
        print("답변:", result["answer"])

In [ ]:
import pandas as pd

golden_path = "golden_dataset_v2.csv"
df_golden = pd.read_csv(golden_path)

print(df_golden.shape)
df_golden.head()

In [ ]:
df_sample = (
    df_golden
    .groupby("question_type", group_keys=False)
    .head(1)
    .head(5)
    .copy()
)

df_sample[["id", "doc_id", "발주기관", "사업명", "question", "answer", "question_type"]]

In [ ]:
def build_search_query(row):
    return f"{row['발주기관']} {row['사업명']} {row['question']}"

In [ ]:
import pandas as pd
import time

test_models = [
    "qwen2.5:3b",
    "gemma3:4b",
    "exaone3.5:2.4b",
]

results = []

for _, row in df_sample.iterrows():
    question = row["question"]
    search_query = build_search_query(row)

    retrieved_docs = retriever.invoke(search_query)
    context = format_rag_context(retrieved_docs)

    retrieved_doc_ids = [
        doc.metadata.get("doc_id", "<unknown>")
        for doc in retrieved_docs
    ]

    retrieved_file_names = [
        doc.metadata.get("file_name", "<unknown>")
        for doc in retrieved_docs
    ]

    retrieved_headers = [
        doc.metadata.get("header_path", "<unknown>")
        for doc in retrieved_docs
    ]

    for model in test_models:
        print("=" * 100)
        print("golden_id:", row["id"])
        print("model:", model)
        print("question:", question)

        prompt = build_rag_prompt(question, context)
        result = ask_ollama(model, prompt)

        results.append({
            "golden_id": row["id"],
            "golden_doc_id": row["doc_id"],
            "발주기관": row["발주기관"],
            "사업명": row["사업명"],
            "question": question,
            "golden_answer": row["answer"],
            "question_type": row["question_type"],
            "eval_metrics": row["eval_metrics"],
            "difficulty": row["difficulty"],
            "source_ref": row["source_ref"],
            "model": model,
            "model_answer": result["answer"],
            "elapsed_sec": result["elapsed_sec"],
            "retrieved_doc_ids": retrieved_doc_ids,
            "retrieved_file_names": retrieved_file_names,
            "retrieved_headers": retrieved_headers,
            "context_preview": context[:1200],
        })

        print("elapsed_sec:", result["elapsed_sec"])
        print("answer:", result["answer"])

df_rag_golden_sample = pd.DataFrame(results)
df_rag_golden_sample

In [ ]:
df_eval = df_rag_golden_sample.copy()

eval_cols = [
    "retrieval_ok",
    "answer_correct",
    "grounded",
    "no_hallucination",
    "format_followed",
    "metadata_used",
    "memo",
]

for col in eval_cols:
    if col not in df_eval.columns:
        df_eval[col] = ""

df_eval[
    [
        "golden_id",
        "question_type",
        "question",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "retrieval_ok",
        "answer_correct",
        "grounded",
        "no_hallucination",
        "format_followed",
        "metadata_used",
        "memo",
    ]
]

In [ ]:
def set_eval(golden_id, model, retrieval_ok, answer_correct, grounded,
             no_hallucination, format_followed, metadata_used, memo):
    mask = (df_eval["golden_id"] == golden_id) & (df_eval["model"] == model)
    df_eval.loc[mask, "retrieval_ok"] = retrieval_ok
    df_eval.loc[mask, "answer_correct"] = answer_correct
    df_eval.loc[mask, "grounded"] = grounded
    df_eval.loc[mask, "no_hallucination"] = no_hallucination
    df_eval.loc[mask, "format_followed"] = format_followed
    df_eval.loc[mask, "metadata_used"] = metadata_used
    df_eval.loc[mask, "memo"] = memo


# Q001: 고려대 사업 목적
set_eval(
    "Q001", "qwen2.5:3b",
    "O", "O", "O", "O", "△", "△",
    "사업 목적은 잘 답변했으나, 질문하지 않은 사업금액과 <unknown>이 불필요하게 포함됨."
)

set_eval(
    "Q001", "gemma3:4b",
    "O", "O", "O", "O", "△", "△",
    "사업 목적은 적절히 답변했으나, 질문하지 않은 사업금액을 함께 언급함."
)

set_eval(
    "Q001", "exaone3.5:2.4b",
    "O", "O", "O", "O", "O", "△",
    "사업 목적을 구조화하여 잘 답변함. 다만 질문하지 않은 예산 정보를 일부 포함함."
)


# Q003: 주요 기능 요구사항
set_eval(
    "Q003", "qwen2.5:3b",
    "O", "△", "△", "O", "△", "△",
    "일부 기능 요구사항을 제시했으나, 검색 기능/성능 요구 등 주변 요구사항이 섞이고 마지막에 사업금액 관련 문장이 부자연스럽게 포함됨."
)

set_eval(
    "Q003", "gemma3:4b",
    "O", "O", "O", "O", "O", "X",
    "포털공통, 마이페이지 등 주요 기능 요구사항을 비교적 간결하게 제시함."
)

set_eval(
    "Q003", "exaone3.5:2.4b",
    "O", "O", "O", "O", "△", "△",
    "주요 기능 요구사항을 풍부하게 제시했으나, 마지막에 사업금액 정보를 불필요하게 덧붙임."
)


# Q010: 고려대 vs 광주과기원 비교
set_eval(
    "Q010", "qwen2.5:3b",
    "X", "△", "O", "O", "O", "△",
    "검색 결과가 고려대 문서에만 치우쳐 광주과기원 근거를 확보하지 못함. 모델은 정보 부족을 비교적 안전하게 처리함."
)

set_eval(
    "Q010", "gemma3:4b",
    "X", "△", "O", "O", "△", "△",
    "검색 결과가 고려대 문서에만 치우쳐 비교가 불가능했으며, 고려대 금액 중심으로 제한적으로 답변함."
)

set_eval(
    "Q010", "exaone3.5:2.4b",
    "X", "X", "△", "X", "△", "△",
    "검색 결과에 광주과기원 근거가 없는데도 광주과기원 목표를 추정하여 답변함. 근거 부족 상황에서 추정 표현이 포함됨."
)


# Q013: 콘텐츠 개발 관리 요구사항
set_eval(
    "Q013", "qwen2.5:3b",
    "O", "△", "△", "△", "△", "△",
    "콘텐츠 개발·관리 요구사항을 다수 제시했으나 반복 표현이 많고, 마지막에 400,000,000원이라는 다른 문서의 금액이 섞인 것으로 보임."
)

set_eval(
    "Q013", "gemma3:4b",
    "O", "O", "O", "O", "O", "X",
    "개발 범위, 자기개발, 비정형콘텐츠, 수정·보완 관리 내용을 근거 기반으로 잘 요약함."
)

set_eval(
    "Q013", "exaone3.5:2.4b",
    "O", "O", "O", "O", "O", "△",
    "콘텐츠 개발 및 관리 요구사항을 상세히 정리했고 사업금액도 해당 문서 기준으로 적절히 제시함."
)


# Q014: 교육/학습 관련 다른 기관 사업
set_eval(
    "Q014", "qwen2.5:3b",
    "△", "X", "△", "O", "△", "X",
    "검색 결과에 D009가 일부 포함되었지만 이를 활용하지 못하고 확인 불가로 답변함."
)

set_eval(
    "Q014", "gemma3:4b",
    "△", "△", "O", "O", "O", "O",
    "검색 결과에 포함된 스포츠윤리센터 LMS 기능개선 사업을 언급함. 다만 golden answer가 기대하는 여러 기관을 모두 포괄하지는 못함."
)

set_eval(
    "Q014", "exaone3.5:2.4b",
    "△", "X", "△", "O", "△", "X",
    "검색 결과에 다른 기관 문서가 일부 포함되었지만 이를 활용하지 못하고 정보 부족으로 처리함."
)

df_eval[
    [
        "golden_id",
        "model",
        "retrieval_ok",
        "answer_correct",
        "grounded",
        "no_hallucination",
        "format_followed",
        "metadata_used",
        "memo",
    ]
]

In [ ]:
score_map = {
    "O": 1.0,
    "△": 0.5,
    "X": 0.0,
    "": None,
}

score_cols = [
    "retrieval_ok",
    "answer_correct",
    "grounded",
    "no_hallucination",
    "format_followed",
    "metadata_used",
]

df_score = df_eval.copy()

for col in score_cols:
    df_score[col + "_score"] = df_score[col].map(score_map)

summary_cols = [col + "_score" for col in score_cols]

model_summary = (
    df_score
    .groupby("model")[summary_cols + ["elapsed_sec"]]
    .mean()
    .reset_index()
)

model_summary

In [ ]:
df_eval.to_csv("rag_golden_sample_eval.csv", index=False, encoding="utf-8-sig")
model_summary.to_csv("rag_golden_sample_model_summary.csv", index=False, encoding="utf-8-sig")

print("저장 완료")

In [ ]:
def retrieve_multi_query(queries, k_each=3):
    all_docs = []
    seen = set()

    for q in queries:
        docs = retriever.invoke(q)

        for doc in docs[:k_each]:
            meta = doc.metadata
            key = (
                meta.get("doc_id", ""),
                meta.get("section_id", ""),
                meta.get("block_id", ""),
            )

            if key not in seen:
                all_docs.append(doc)
                seen.add(key)

    return all_docs

In [ ]:
q010_queries = [
    "고려대학교 차세대 포털 학사 정보시스템 구축사업 사업목적 사업금액 추진목표",
    "광주과학기술원 학사시스템 기능개선 사업 사업목적 사업금액 추진목표",
]

retrieved_docs = retrieve_multi_query(q010_queries, k_each=3)
context = format_rag_context(retrieved_docs)

print([doc.metadata.get("doc_id") for doc in retrieved_docs])
print(context[:3000])

In [ ]:
question = "고려대학교 차세대 포털 사업과 광주과학기술원 학사시스템 사업을 비교해줘"

q010_queries = [
    "고려대학교 차세대 포털 학사 정보시스템 구축사업 사업목적 사업금액 추진목표",
    "광주과학기술원 학사시스템 기능개선 사업 사업목적 사업금액 추진목표",
]

retrieved_docs = retrieve_multi_query(q010_queries, k_each=3)
context = format_rag_context(retrieved_docs)

print([doc.metadata.get("doc_id") for doc in retrieved_docs])
print(context[:3000])

In [ ]:
q010_rerun_results = []

for model in test_models:
    prompt = build_rag_prompt(question, context)
    result = ask_ollama(model, prompt)

    q010_rerun_results.append({
        "golden_id": "Q010",
        "model": model,
        "question": question,
        "answer": result["answer"],
        "elapsed_sec": result["elapsed_sec"],
        "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
        "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
        "context_preview": context[:1200],
    })

    print("=" * 100)
    print("MODEL:", model)
    print("elapsed_sec:", result["elapsed_sec"])
    print(result["answer"])

df_q010_rerun = pd.DataFrame(q010_rerun_results)
df_q010_rerun

In [ ]:
q014_queries = [
    "국민연금공단 2024년 이러닝시스템 운영 용역 교육 학습",
    "스포츠윤리센터 LMS 학습지원시스템 기능개선 교육 학습",
    "고려대학교 차세대 포털 학사 정보시스템 교육 학습",
    "광주과학기술원 학사시스템 기능개선 교육 학습",
]

retrieved_docs = retrieve_multi_query(q014_queries, k_each=2)
context = format_rag_context(retrieved_docs)

print([doc.metadata.get("doc_id") for doc in retrieved_docs])
print(context[:3000])

In [ ]:
question = "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?"

q014_queries = [
    "국민연금공단 2024년 이러닝시스템 운영 용역 교육 학습",
    "스포츠윤리센터 LMS 학습지원시스템 기능개선 교육 학습",
    "고려대학교 차세대 포털 학사 정보시스템 교육 학습",
    "광주과학기술원 학사시스템 기능개선 교육 학습",
]

retrieved_docs = retrieve_multi_query(q014_queries, k_each=2)
context = format_rag_context(retrieved_docs)

q014_rerun_results = []

for model in test_models:
    prompt = build_rag_prompt(question, context)
    result = ask_ollama(model, prompt)

    q014_rerun_results.append({
        "golden_id": "Q014",
        "model": model,
        "question": question,
        "answer": result["answer"],
        "elapsed_sec": result["elapsed_sec"],
        "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
        "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
        "context_preview": context[:1200],
    })

    print("=" * 100)
    print("MODEL:", model)
    print("elapsed_sec:", result["elapsed_sec"])
    print(result["answer"])

df_q014_rerun = pd.DataFrame(q014_rerun_results)
df_q014_rerun

In [ ]:
def build_multi_doc_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색된 문서 근거]에는 여러 기관의 사업 정보가 포함되어 있습니다.
각 검색결과의 [문서명], [사업명], [발주기관], [사업금액], [문서내용]을 모두 확인한 뒤 답변하세요.

[검색된 문서 근거]
{context}

[질문]
{question}

[답변 작성 규칙]
- 반드시 한국어 존댓말로 답변하세요.
- 문서 근거에 있는 내용만 사용하세요.
- 질문이 여러 기관 또는 다른 기관의 사례를 묻는 경우, 검색결과에 등장한 서로 다른 발주기관을 빠짐없이 확인하세요.
- "다른 기관"은 검색결과에 포함된 국민연금공단, 재단법인스포츠윤리센터, 고려대학교, 광주과학기술원 등 서로 다른 발주기관을 의미합니다.
- 교육, 학습, 이러닝, LMS, 학사시스템, 포털, 학습관리시스템과 관련된 사업이 있으면 기관별로 나열하세요.
- 각 항목에는 발주기관, 사업명, 관련 근거, 사업금액을 포함하세요.
- 근거가 있는 사업을 누락하지 마세요.
- 질문과 관련 없는 사업은 제외하세요.
- 근거가 부족한 경우에만 정보 부족이라고 답변하세요.
- 추정하지 마세요.
"""

In [ ]:
question = "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?"

q014_queries = [
    "국민연금공단 2024년 이러닝시스템 운영 용역 교육 학습",
    "스포츠윤리센터 LMS 학습지원시스템 기능개선 교육 학습",
    "고려대학교 차세대 포털 학사 정보시스템 교육 학습 LMS",
    "광주과학기술원 학사시스템 기능개선 교육 학습",
]

retrieved_docs = retrieve_multi_query(q014_queries, k_each=2)
context = format_rag_context(retrieved_docs)

q014_rerun_results_v2 = []

for model in test_models:
    prompt = build_multi_doc_prompt(question, context)
    result = ask_ollama(model, prompt)

    q014_rerun_results_v2.append({
        "golden_id": "Q014",
        "model": model,
        "question": question,
        "answer": result["answer"],
        "elapsed_sec": result["elapsed_sec"],
        "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
        "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
        "context_preview": context[:1200],
    })

    print("=" * 100)
    print("MODEL:", model)
    print("elapsed_sec:", result["elapsed_sec"])
    print(result["answer"])

df_q014_rerun_v2 = pd.DataFrame(q014_rerun_results_v2)
df_q014_rerun_v2

In [ ]:
# Q014 v2 재실행 결과 기준 평가 갱신

set_eval(
    "Q014", "qwen2.5:3b",
    "O", "X", "△", "X", "△", "△",
    "다중 query 검색으로 관련 문서는 모두 확보했으나, 사업명과 사업금액을 잘못 생성함. 국민연금공단 773,801,000원을 2천만원으로, 스포츠윤리센터 46,445,000원을 1천만원으로 오답 처리함."
)

set_eval(
    "Q014", "gemma3:4b",
    "O", "O", "O", "O", "O", "O",
    "국민연금공단, 스포츠윤리센터, 고려대학교, 광주과학기술원 4개 기관의 교육/학습 관련 사업을 모두 정확히 제시하고 사업금액도 원문 그대로 유지함."
)

set_eval(
    "Q014", "exaone3.5:2.4b",
    "O", "△", "O", "O", "O", "O",
    "스포츠윤리센터, 고려대학교, 광주과학기술원은 정확히 제시했으나 국민연금공단 이러닝시스템 운영 용역을 누락함."
)

df_eval[
    [
        "golden_id",
        "model",
        "retrieval_ok",
        "answer_correct",
        "grounded",
        "no_hallucination",
        "format_followed",
        "metadata_used",
        "memo",
    ]
]

In [ ]:
df_score = df_eval.copy()

for col in score_cols:
    df_score[col + "_score"] = df_score[col].map(score_map)

summary_cols = [col + "_score" for col in score_cols]

model_summary_v2 = (
    df_score
    .groupby("model")[summary_cols + ["elapsed_sec"]]
    .mean()
    .reset_index()
)

model_summary_v2

In [ ]:
df_eval.to_csv("rag_golden_sample_eval_v2.csv", index=False, encoding="utf-8-sig")
model_summary_v2.to_csv("rag_golden_sample_model_summary_v2.csv", index=False, encoding="utf-8-sig")

print("저장 완료")

5. Golden QA 기반 소형 3종 실행

| 지표 | O | △ | X |
|---|---|---|---|
| 점수 | 1.0 | 0.5 | 0.0 |

In [ ]:
def is_multi_doc_question(row):
    text = str(row.get("question_type", "")) + " " + str(row.get("doc_id", "")) + " " + str(row.get("question", ""))
    
    multi_keywords = [
        "다중문서",
        "비교",
        "종합",
        "_vs_",
        "복수",
        "다른 기관",
    ]
    
    return any(keyword in text for keyword in multi_keywords)

In [ ]:
def build_queries_for_row(row):
    question = row["question"]
    org = str(row.get("발주기관", ""))
    project = str(row.get("사업명", ""))
    doc_id = str(row.get("doc_id", ""))

    # Q010처럼 비교 대상이 명확한 경우
    if "고려대" in question or "고려대학교" in question or "광주과학기술원" in question:
        if "비교" in question:
            return [
                "고려대학교 차세대 포털 학사 정보시스템 구축사업 사업목적 사업금액 추진목표",
                "광주과학기술원 학사시스템 기능개선 사업 사업목적 사업금액 추진목표",
            ]

    # Q014처럼 교육/학습 관련 다중문서 종합
    if "교육" in question or "학습" in question or "이러닝" in question or "LMS" in question:
        if "다른 기관" in question or "없나" in question:
            return [
                "국민연금공단 2024년 이러닝시스템 운영 용역 교육 학습",
                "스포츠윤리센터 LMS 학습지원시스템 기능개선 교육 학습",
                "고려대학교 차세대 포털 학사 정보시스템 교육 학습 LMS",
                "광주과학기술원 학사시스템 기능개선 교육 학습",
            ]

    # 기본 다중문서 처리
    if is_multi_doc_question(row):
        return [
            f"{org} {project} {question}",
            question,
        ]

    # 단일문서 처리
    return [
        f"{org} {project} {question}"
    ]

In [ ]:
import pandas as pd
import time

test_models = [
    "qwen2.5:3b",
    "gemma3:4b",
    "exaone3.5:2.4b",
]

all_results = []

# 전체 실행
df_run = df_golden.copy()

# 빠른 테스트만 하고 싶으면 아래 사용
# df_run = df_golden.head(30).copy()

for idx, row in df_run.iterrows():
    question = row["question"]
    queries = build_queries_for_row(row)

    if len(queries) > 1:
        retrieved_docs = retrieve_multi_query(queries, k_each=3)
    else:
        retrieved_docs = retriever.invoke(queries[0])

    context = format_rag_context(retrieved_docs)

    # 다중문서면 전용 프롬프트, 아니면 일반 프롬프트
    use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

    for model in test_models:
        if use_multi_prompt:
            prompt = build_multi_doc_prompt(question, context)
        else:
            prompt = build_rag_prompt(question, context)

        result = ask_ollama(model, prompt)

        all_results.append({
            "golden_id": row["id"],
            "golden_doc_id": row["doc_id"],
            "발주기관": row["발주기관"],
            "사업명": row["사업명"],
            "question": question,
            "golden_answer": row["answer"],
            "question_type": row["question_type"],
            "eval_metrics": row["eval_metrics"],
            "difficulty": row["difficulty"],
            "source_ref": row["source_ref"],
            "model": model,
            "model_answer": result["answer"],
            "elapsed_sec": result["elapsed_sec"],
            "queries": queries,
            "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
            "retrieved_file_names": [doc.metadata.get("file_name") for doc in retrieved_docs],
            "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
            "context_preview": context[:1200],
        })

    print(f"[{idx+1}/{len(df_run)}] 완료:", row["id"], question)

df_rag_golden_all = pd.DataFrame(all_results)
df_rag_golden_all

In [ ]:
df_rag_golden_all.to_csv(
    "rag_golden_all_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df_rag_golden_all.shape)

In [ ]:
df_fact = df_rag_golden_all[
    df_rag_golden_all["golden_answer"].notna()
].copy()

df_prompt_eval = df_rag_golden_all[
    df_rag_golden_all["golden_answer"].isna()
].copy()

print("정답 있는 QA:", df_fact.shape)
print("프롬프트/질문재작성 평가:", df_prompt_eval.shape)

In [ ]:
time_summary = (
    df_rag_golden_all
    .groupby("model")["elapsed_sec"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

time_summary

In [ ]:
type_time_summary = (
    df_rag_golden_all
    .groupby(["question_type", "model"])["elapsed_sec"]
    .mean()
    .reset_index()
)

type_time_summary

In [ ]:
df_rag_golden_all[[
    "golden_id",
    "question_type",
    "question",
    "model",
    "retrieved_doc_ids",
    "model_answer"
]].head()

In [ ]:
df_multi = df_rag_golden_all[
    df_rag_golden_all["question_type"].astype(str).str.contains("다중|비교|종합", na=False)
].copy()

df_multi[[
    "golden_id",
    "question",
    "model",
    "retrieved_doc_ids",
    "model_answer"
]].head(30)

In [ ]:
df_eval_sample = (
    df_rag_golden_all
    .groupby(["question_type", "model"], group_keys=False)
    .head(3)
    .copy()
)

df_eval_sample.shape

In [ ]:
eval_cols = [
    "retrieval_ok",
    "answer_correct",
    "grounded",
    "no_hallucination",
    "format_followed",
    "metadata_used",
    "memo",
]

for col in eval_cols:
    df_eval_sample[col] = ""

df_eval_sample.to_csv(
    "rag_golden_eval_sample_for_manual.csv",
    index=False,
    encoding="utf-8-sig"
)

df_eval_sample

In [ ]:
df_short_answers = df_rag_golden_all[
    df_rag_golden_all["model_answer"].astype(str).str.len() < 20
].copy()

df_short_answers[[
    "golden_id",
    "question",
    "model",
    "model_answer",
    "elapsed_sec"
]]

In [ ]:
print("전체 결과 shape:", df_rag_golden_all.shape)

print("\n모델별 row 수")
print(df_rag_golden_all["model"].value_counts())

print("\nquestion_type별 row 수")
print(df_rag_golden_all["question_type"].value_counts(dropna=False))

In [ ]:
error_mask = (
    df_rag_golden_all["model_answer"].astype(str).str.contains("Ollama 서버에 연결하지 못했습니다", na=False)
    | df_rag_golden_all["model_answer"].astype(str).str.contains("오류 발생", na=False)
    | df_rag_golden_all["model_answer"].astype(str).str.contains("HTTP", na=False)
    | (df_rag_golden_all["elapsed_sec"].fillna(-1) < 0)
)

df_errors = df_rag_golden_all[error_mask].copy()

df_errors[
    ["golden_id", "question", "model", "model_answer", "elapsed_sec", "retrieved_doc_ids"]
]

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
import re

def is_short_bad_answer(x):
    x = str(x).strip()

    if len(x) >= 20:
        return False

    # 정상 가능성이 있는 짧은 답변 제외
    if x == "<unknown>":
        return False

    # 금액/날짜 단답은 정상일 수 있음
    if re.search(r"\d", x):
        return False

    return True

short_bad_mask = df_rag_golden_all["model_answer"].apply(is_short_bad_answer)

df_short_bad = df_rag_golden_all[short_bad_mask].copy()

df_short_bad[
    ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
]

In [ ]:
df_rerun_targets = pd.concat(
    [df_errors, df_short_bad],
    axis=0
).drop_duplicates(
    subset=["golden_id", "model"]
).copy()

df_rerun_targets[
    ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
]

In [ ]:
def rerun_one_row(row):
    question = row["question"]
    queries = build_queries_for_row(row)

    if len(queries) > 1:
        retrieved_docs = retrieve_multi_query(queries, k_each=3)
    else:
        retrieved_docs = retriever.invoke(queries[0])

    context = format_rag_context(retrieved_docs)

    use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

    if use_multi_prompt:
        prompt = build_multi_doc_prompt(question, context)
    else:
        prompt = build_rag_prompt(question, context)

    result = ask_ollama(row["model"], prompt)

    updated = row.copy()
    updated["model_answer"] = result["answer"]
    updated["elapsed_sec"] = result["elapsed_sec"]
    updated["queries"] = queries
    updated["retrieved_doc_ids"] = [doc.metadata.get("doc_id") for doc in retrieved_docs]
    updated["retrieved_file_names"] = [doc.metadata.get("file_name") for doc in retrieved_docs]
    updated["retrieved_headers"] = [doc.metadata.get("header_path") for doc in retrieved_docs]
    updated["context_preview"] = context[:1200]

    return updated

In [ ]:
rerun_rows = []

for i, (_, row) in enumerate(df_rerun_targets.iterrows(), 1):
    print(f"[{i}/{len(df_rerun_targets)}] 재실행:", row["golden_id"], row["model"], row["question"])
    updated_row = rerun_one_row(row)
    rerun_rows.append(updated_row)

df_rerun_fixed = pd.DataFrame(rerun_rows)
df_rerun_fixed[
    ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
]

In [ ]:
df_fixed = df_rag_golden_all.copy()

for _, row in df_rerun_fixed.iterrows():
    mask = (
        (df_fixed["golden_id"] == row["golden_id"])
        & (df_fixed["model"] == row["model"])
    )

    target_indices = df_fixed.index[mask]

    if len(target_indices) == 0:
        print("매칭되는 row 없음:", row["golden_id"], row["model"])
        continue

    if len(target_indices) > 1:
        print("중복 매칭 row 있음:", row["golden_id"], row["model"], list(target_indices))

    target_idx = target_indices[0]

    for col in df_fixed.columns:
        if col in row.index:
            df_fixed.at[target_idx, col] = row[col]

print(df_fixed.shape)

In [ ]:
error_mask_fixed = (
    df_fixed["model_answer"].astype(str).str.contains("Ollama 서버에 연결하지 못했습니다", na=False)
    | df_fixed["model_answer"].astype(str).str.contains("오류 발생", na=False)
    | df_fixed["model_answer"].astype(str).str.contains("HTTP", na=False)
    | (df_fixed["elapsed_sec"].fillna(-1) < 0)
)

short_bad_mask_fixed = df_fixed["model_answer"].apply(is_short_bad_answer)

print("오류 row 수:", error_mask_fixed.sum())
print("짧은 비정상 답변 row 수:", short_bad_mask_fixed.sum())

df_fixed[error_mask_fixed | short_bad_mask_fixed][
    ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
]

In [ ]:
df_valid_time = df_fixed[
    (~error_mask_fixed)
    & (df_fixed["elapsed_sec"].notna())
    & (df_fixed["elapsed_sec"] >= 0)
].copy()

time_summary_fixed = (
    df_valid_time
    .groupby("model")["elapsed_sec"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

time_summary_fixed

6. Gemma 안정성 검증

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
def get_model_options(model):
    """
    모델별 Ollama generation 옵션 설정
    """
    options = {
        "temperature": 0.1,
        "num_predict": 1024,
    }

    # 이전 실험 인사이트 반영
    if model == "gemma3:4b":
        options.update({
            "top_k": 2,
            "top_p": 0.9,
        })

    return options

In [ ]:
# [정리] 아래 함수는 이후 셀에서 재정의됨. 실행 불필요.
import requests
import time

def ask_ollama(model, prompt, max_retries=3, sleep_sec=2):
    """
    Ollama 호출 함수
    - 모델별 options 적용
    - gemma3:4b top_k=2 반영
    - 재시도 처리
    - perf_counter로 음수 시간 방지
    """
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": get_model_options(model),
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        start_time = time.perf_counter()

        try:
            response = requests.post(url, json=payload, timeout=300)
            elapsed = max(round(time.perf_counter() - start_time, 2), 0)

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code} 오류: {response.text}"
                print(f"[{model}] attempt {attempt} 실패:", last_error)
                time.sleep(sleep_sec)
                continue

            result = response.json()
            answer = result.get("response", "").strip()

            return {
                "model": model,
                "answer": answer,
                "elapsed_sec": elapsed,
                "attempt": attempt,
            }

        except requests.exceptions.ConnectionError:
            last_error = "Ollama 서버에 연결하지 못했습니다. `ollama serve` 실행 여부를 확인하세요."
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except requests.exceptions.ReadTimeout:
            last_error = "오류 발생: ReadTimeout"
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except Exception as e:
            last_error = f"오류 발생: {type(e).__name__} - {e}"
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

    return {
        "model": model,
        "answer": last_error,
        "elapsed_sec": None,
        "attempt": max_retries,
    }

In [ ]:
import pandas as pd
import time

def run_all_for_one_model(model, df_run, output_path, sleep_sec=0.5):
    results = []

    total = len(df_run)

    for idx, row in df_run.iterrows():
        question = row["question"]
        queries = build_queries_for_row(row)

        if len(queries) > 1:
            retrieved_docs = retrieve_multi_query(queries, k_each=3)
        else:
            retrieved_docs = retriever.invoke(queries[0])

        context = format_rag_context(retrieved_docs)

        use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

        if use_multi_prompt:
            prompt = build_multi_doc_prompt(question, context)
        else:
            prompt = build_rag_prompt(question, context)

        result = ask_ollama(model, prompt)

        results.append({
            "golden_id": row["id"],
            "golden_doc_id": row["doc_id"],
            "발주기관": row["발주기관"],
            "사업명": row["사업명"],
            "question": question,
            "golden_answer": row["answer"],
            "question_type": row["question_type"],
            "eval_metrics": row["eval_metrics"],
            "difficulty": row["difficulty"],
            "source_ref": row["source_ref"],
            "model": model,
            "model_answer": result["answer"],
            "elapsed_sec": result["elapsed_sec"],
            "attempt": result.get("attempt"),
            "queries": queries,
            "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
            "retrieved_file_names": [doc.metadata.get("file_name") for doc in retrieved_docs],
            "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
            "context_preview": context[:1200],
        })

        print(f"[{idx + 1}/{total}] {model} 완료:", row["id"], "|", question)

        time.sleep(sleep_sec)

        # 중간 저장
        if (idx + 1) % 10 == 0:
            pd.DataFrame(results).to_csv(
                output_path,
                index=False,
                encoding="utf-8-sig"
            )

    df_result = pd.DataFrame(results)
    df_result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("저장 완료:", output_path, df_result.shape)

    return df_result

In [ ]:
df_qwen_all = run_all_for_one_model(
    model="qwen2.5:3b",
    df_run=df_golden,
    output_path="rag_golden_qwen2_5_3b_results_stable.csv",
    sleep_sec=0.5,
)

In [ ]:
def check_result_quality(df):
    error_mask = (
        df["model_answer"].astype(str).str.contains("Ollama 서버에 연결하지 못했습니다", na=False)
        | df["model_answer"].astype(str).str.contains("오류 발생", na=False)
        | df["model_answer"].astype(str).str.contains("HTTP", na=False)
        | df["elapsed_sec"].isna()
        | (df["elapsed_sec"].fillna(-1) < 0)
    )

    short_bad_mask = df["model_answer"].apply(is_short_bad_answer)

    print("전체 row:", len(df))
    print("오류 row:", error_mask.sum())
    print("짧은 비정상 답변 row:", short_bad_mask.sum())

    return df[error_mask | short_bad_mask][
        ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
    ]

check_result_quality(df_qwen_all)

In [ ]:
df_exaone_all = run_all_for_one_model(
    model="exaone3.5:2.4b",
    df_run=df_golden,
    output_path="rag_golden_exaone3_5_2_4b_results_stable.csv",
    sleep_sec=0.5,
)

In [ ]:
check_result_quality(df_exaone_all)

In [ ]:
df_gemma_all = run_all_for_one_model(
    model="gemma3:4b",
    df_run=df_golden,
    output_path="rag_golden_gemma3_4b_topk2_results_stable.csv",
    sleep_sec=0.8,
)

In [ ]:
check_result_quality(df_gemma_all)

In [ ]:
def get_model_options(model, mode="default"):
    """
    모델별 generation 옵션
    - gemma3:4b는 기본적으로 top_k=2 적용
    - 짧은 출력 발생 시 retry_relaxed / retry_strict 모드 사용
    """
    options = {
        "temperature": 0.1,
        "num_predict": 1024,
    }

    if model == "gemma3:4b":
        if mode == "default":
            options.update({
                "top_k": 2,
                "top_p": 0.9,
                "repeat_penalty": 1.1,
            })

        elif mode == "retry_relaxed":
            # top_k=2가 너무 좁아서 EOS/짧은 출력으로 빠질 가능성 대응
            options.update({
                "top_k": 20,
                "top_p": 0.95,
                "temperature": 0.2,
                "repeat_penalty": 1.1,
            })

        elif mode == "retry_strict":
            # 답변 길이를 확보하기 위한 마지막 재시도
            options.update({
                "top_k": 40,
                "top_p": 0.95,
                "temperature": 0.2,
                "repeat_penalty": 1.05,
            })

    return options

In [ ]:
import requests
import time

def ask_ollama(model, prompt, mode="default", max_retries=3, sleep_sec=2):
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": get_model_options(model, mode=mode),
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        start_time = time.perf_counter()

        try:
            response = requests.post(url, json=payload, timeout=300)
            elapsed = max(round(time.perf_counter() - start_time, 2), 0)

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code} 오류: {response.text}"
                print(f"[{model} / {mode}] attempt {attempt} 실패:", last_error)
                time.sleep(sleep_sec)
                continue

            result = response.json()
            answer = result.get("response", "").strip()

            return {
                "model": model,
                "answer": answer,
                "elapsed_sec": elapsed,
                "attempt": attempt,
                "mode": mode,
            }

        except requests.exceptions.ConnectionError:
            last_error = "Ollama 서버에 연결하지 못했습니다. `ollama serve` 실행 여부를 확인하세요."
            print(f"[{model} / {mode}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except requests.exceptions.ReadTimeout:
            last_error = "오류 발생: ReadTimeout"
            print(f"[{model} / {mode}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except Exception as e:
            last_error = f"오류 발생: {type(e).__name__} - {e}"
            print(f"[{model} / {mode}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

    return {
        "model": model,
        "answer": last_error,
        "elapsed_sec": None,
        "attempt": max_retries,
        "mode": mode,
    }

In [ ]:
import re

def is_short_bad_answer(x):
    x = str(x).strip()

    if len(x) >= 20:
        return False

    if x == "<unknown>":
        return False

    if re.search(r"\d", x):
        return False

    return True

In [ ]:
def strengthen_prompt_for_gemma(prompt):
    return prompt + """

[Gemma 응답 보강 규칙]
- 답변은 반드시 완성된 문장으로 작성하세요.
- 한 단어, 한 글자, 제목 기호만 출력하지 마세요.
- 최소 3문장 이상으로 답변하세요.
- 근거가 부족한 경우에도 "문서 근거에서 확인 가능한 정보가 부족합니다"라고 완성된 문장으로 답변하세요.
"""

In [ ]:
def rerun_one_row_gemma_with_retry(row):
    question = row["question"]
    queries = build_queries_for_row(row)

    if len(queries) > 1:
        retrieved_docs = retrieve_multi_query(queries, k_each=3)
    else:
        retrieved_docs = retriever.invoke(queries[0])

    context = format_rag_context(retrieved_docs)

    use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

    if use_multi_prompt:
        prompt = build_multi_doc_prompt(question, context)
    else:
        prompt = build_rag_prompt(question, context)

    # 1차: 기존 인사이트 top_k=2
    result = ask_ollama("gemma3:4b", prompt, mode="default")

    # 2차: 짧으면 top_k 완화
    if is_short_bad_answer(result["answer"]):
        print("짧은 출력 감지 → retry_relaxed:", row["golden_id"], result["answer"])
        result = ask_ollama("gemma3:4b", prompt, mode="retry_relaxed")

    # 3차: 그래도 짧으면 프롬프트 보강
    if is_short_bad_answer(result["answer"]):
        print("짧은 출력 반복 → retry_strict + prompt 강화:", row["golden_id"], result["answer"])
        strengthened_prompt = strengthen_prompt_for_gemma(prompt)
        result = ask_ollama("gemma3:4b", strengthened_prompt, mode="retry_strict")

    updated = row.copy()
    updated["model_answer"] = result["answer"]
    updated["elapsed_sec"] = result["elapsed_sec"]
    updated["attempt"] = result.get("attempt")
    updated["gemma_retry_mode"] = result.get("mode")
    updated["queries"] = queries
    updated["retrieved_doc_ids"] = [doc.metadata.get("doc_id") for doc in retrieved_docs]
    updated["retrieved_file_names"] = [doc.metadata.get("file_name") for doc in retrieved_docs]
    updated["retrieved_headers"] = [doc.metadata.get("header_path") for doc in retrieved_docs]
    updated["context_preview"] = context[:1200]

    return updated

In [ ]:
gemma_short_mask = df_gemma_all["model_answer"].apply(is_short_bad_answer)

df_gemma_rerun_targets = df_gemma_all[gemma_short_mask].copy()

print("gemma 재실행 대상 수:", len(df_gemma_rerun_targets))

df_gemma_rerun_targets[
    ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
]

In [ ]:
gemma_rerun_rows = []

for i, (_, row) in enumerate(df_gemma_rerun_targets.iterrows(), 1):
    print(f"\n[{i}/{len(df_gemma_rerun_targets)}] gemma 재실행:", row["golden_id"], row["question"])

    updated_row = rerun_one_row_gemma_with_retry(row)
    gemma_rerun_rows.append(updated_row)

    time.sleep(1)

    pd.DataFrame(gemma_rerun_rows).to_csv(
        "gemma_short_answer_rerun_results.csv",
        index=False,
        encoding="utf-8-sig"
    )

df_gemma_rerun_fixed = pd.DataFrame(gemma_rerun_rows)

df_gemma_rerun_fixed[
    ["golden_id", "question", "model_answer", "elapsed_sec", "gemma_retry_mode"]
]

In [ ]:
import requests
import time

def ask_ollama_chat_gemma(prompt, max_retries=2, sleep_sec=2):
    url = "http://127.0.0.1:11434/api/chat"

    payload = {
        "model": "gemma3:4b",
        "messages": [
            {
                "role": "system",
                "content": "당신은 공공 RFP 문서를 분석하는 AI입니다. 반드시 한국어 존댓말로 완성된 문장으로 답변하세요."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0.1,
            "top_k": 2,
            "top_p": 0.9,
            "num_predict": 1024,
            "repeat_penalty": 1.1,
        }
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        start_time = time.perf_counter()

        try:
            response = requests.post(url, json=payload, timeout=300)
            elapsed = max(round(time.perf_counter() - start_time, 2), 0)

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code} 오류: {response.text}"
                print(f"[gemma chat] attempt {attempt} 실패:", last_error)
                time.sleep(sleep_sec)
                continue

            result = response.json()
            answer = result.get("message", {}).get("content", "").strip()

            return {
                "model": "gemma3:4b",
                "answer": answer,
                "elapsed_sec": elapsed,
                "attempt": attempt,
                "api_mode": "chat",
            }

        except Exception as e:
            last_error = f"오류 발생: {type(e).__name__} - {e}"
            print(f"[gemma chat] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

    return {
        "model": "gemma3:4b",
        "answer": last_error,
        "elapsed_sec": None,
        "attempt": max_retries,
        "api_mode": "chat",
    }

In [ ]:
test_ids = ["Q014", "Q045", "Q111"]

df_gemma_chat_test_targets = df_gemma_all[
    df_gemma_all["golden_id"].isin(test_ids)
].copy()

chat_test_rows = []

for _, row in df_gemma_chat_test_targets.iterrows():
    question = row["question"]
    queries = build_queries_for_row(row)

    if len(queries) > 1:
        retrieved_docs = retrieve_multi_query(queries, k_each=3)
    else:
        retrieved_docs = retriever.invoke(queries[0])

    context = format_rag_context(retrieved_docs)

    use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

    if use_multi_prompt:
        prompt = build_multi_doc_prompt(question, context)
    else:
        prompt = build_rag_prompt(question, context)

    print("테스트:", row["golden_id"], question)
    result = ask_ollama_chat_gemma(prompt)

    chat_test_rows.append({
        "golden_id": row["golden_id"],
        "question": question,
        "old_answer": row["model_answer"],
        "chat_answer": result["answer"],
        "elapsed_sec": result["elapsed_sec"],
    })

df_gemma_chat_test = pd.DataFrame(chat_test_rows)
df_gemma_chat_test

In [ ]:
df_rag_golden_all_stable = pd.concat(
    [df_qwen_all, df_gemma_all, df_exaone_all],
    ignore_index=True
)

df_rag_golden_all_stable.to_csv(
    "rag_golden_all_results_stable_by_model.csv",
    index=False,
    encoding="utf-8-sig"
)

df_rag_golden_all_stable.shape

In [ ]:
time_summary_stable = (
    df_rag_golden_all_stable
    .groupby("model")["elapsed_sec"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

time_summary_stable

7. 전체 결과 정제

In [ ]:
def is_generation_error_answer(x):
    x = str(x).strip()

    if "Ollama 서버에 연결하지 못했습니다" in x:
        return True
    if "오류 발생" in x:
        return True
    if "HTTP" in x:
        return True
    if is_short_bad_answer(x):
        return True

    return False


df_rag_golden_all_stable["generation_failed"] = (
    df_rag_golden_all_stable["model_answer"].apply(is_generation_error_answer)
)

failure_summary = (
    df_rag_golden_all_stable
    .groupby("model")["generation_failed"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

failure_summary["failure_rate"] = failure_summary["mean"] * 100

failure_summary

In [ ]:
df_main_compare = df_rag_golden_all_stable[
    df_rag_golden_all_stable["model"] != "gemma3:4b"
].copy()

main_failure_summary = (
    df_main_compare
    .groupby("model")["generation_failed"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

main_failure_summary["failure_rate"] = main_failure_summary["mean"] * 100

main_failure_summary

## RAG 연동 실험 정리

본 노트북에서는 Vector DB 기반 RAG 검색 결과를 Generation 프롬프트에 연결하고, Golden QA 123개를 기준으로 소형 로컬 모델 3종의 응답을 비교하였습니다.

### 최종 응답시간 비교

| 모델 | 평균 | 중앙값 | 최소 | 최대 |
|---|---|---|---|---|
| exaone3.5:2.4b | 2.11초 | 2.01초 | 0.45초 | 5.50초 |
| gemma3:4b | 2.10초 | 1.49초 | 0.74초 | 8.92초 |
| qwen2.5:3b | 2.31초 | 1.73초 | 0.52초 | 9.30초 |

### 생성 실패율

| 모델 | 실패 건수 | 실패율 |
|---|---|---|
| exaone3.5:2.4b | 2 / 123 | 1.63% |
| qwen2.5:3b | 2 / 123 | 1.63% |
| gemma3:4b | 17 / 123 | 13.82% |

### 주요 발견

- gemma3:4b는 한 글자/한 단어 수준의 불완전 출력이 반복 발생하여, retry 및 프롬프트 강화로도 해결되지 않았다.
- exaone3.5:2.4b와 qwen2.5:3b는 실패율이 동일하나, exaone이 응답 안정성 측면에서 소폭 우위를 보였습니다.
- 소형 3종 모델 비교 결과, 생성 실패율이 0%인 모델이 없어 상위 모델(7B 이상) 비교가 필요하다는 결론에 도달하였습니다.

> 상위 모델 비교는 `upper_model_comparison.ipynb`에서 진행합니다.
